[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb)

# Yaya-125M — True AI Curriculum Training

Trains Yaya through the full 16-phase True AI curriculum.
Each phase builds on the last — world knowledge → conversation → reasoning → Swahili → code → alignment.

| Stage | Phases | What it learns | Est. time (T4) |
|-------|--------|----------------|----------------|
| A | 1–4  | World knowledge, conversation, instruction following, Kenya Q&A | ~4 hrs |
| B | 5–8  | Chain-of-thought, math, logic, self-reflection | ~4 hrs |
| C | 9–11 | Tool calling, multi-step tools, RAG grounding | ~3 hrs |
| D | 12–14| Code, structured output, Swahili fluency | ~3 hrs |
| E | 15–16| Safety refusals, DPO preference alignment | ~2 hrs |

**Required:** Runtime → Change runtime type → **T4 GPU** (or better)

Each session auto-resumes — phases already done are skipped. Progress is saved to Google Drive after every phase.

## Step 0 — Check GPU

In [ ]:
import torch, os, sys

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  ({vram} GB VRAM)')
    print(f'PyTorch: {torch.__version__}')
    dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    is_a100 = vram >= 35
    print(f'dtype: {dtype}  |  A100-class: {is_a100}')
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## Step 1 — Mount Drive + clone repo

Checkpoints and progress survive session resets via Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import os, sys, subprocess

REPO = '/content/miss-yaya'
ROOT = f'{REPO}/yaya-ai'

if not os.path.exists(REPO):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/jaylink-coder/miss-yaya.git', REPO], check=True)
else:
    print('Repo already present — pulling latest...')
    result = subprocess.run(['git', 'pull'], cwd=REPO, capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Install dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentencepiece', 'pyyaml', 'huggingface_hub'], check=True)
print('Dependencies ready.')

## Step 2 — Set HuggingFace token

The runner pulls the latest checkpoint from HF Hub and pushes after each phase.

**Recommended:** Add `HF_TOKEN` in Colab Secrets (🔑 icon in left sidebar) so you don't paste it here.

In [ ]:
import os

# Try Colab secrets first
hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
    if hf_token:
        print('HF token loaded from Colab Secrets.')
except Exception:
    pass

if not hf_token:
    # Fallback: paste your token here (don't commit this!)
    hf_token = ''  # ← paste hf_xxx here if secrets not set

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN set.')
else:
    print('WARNING: No HF_TOKEN — Hub pull/push disabled. Checkpoints will only save to Drive.')

## Step 3 — Run curriculum phases

**Run all phases automatically** (picks up where it left off):
```
!python scripts/colab_run_phases.py
```

**Run a specific phase or range:**
```
!python scripts/colab_run_phases.py --phase 1
!python scripts/colab_run_phases.py --phase 1-4
```

Recommended for a single 4-hour Colab session: **phases 1–4** (Foundation block).
Next session: **phases 5–8** (Reasoning block), and so on.

In [ ]:
# Show which phases are already done
import json, os

done_file = '/content/drive/MyDrive/yaya-checkpoints/phase_done.json'
if os.path.exists(done_file):
    with open(done_file) as f:
        done = json.load(f).get('completed', [])
    print(f'Phases already complete: {done}')
    remaining = [i for i in range(1, 17) if i not in done]
    print(f'Remaining: {remaining}')
else:
    print('No phases done yet — starting from Phase 1.')

In [ ]:
# Auto-run next phase
# Change --phase 1-4 to run a specific range, or remove it to run just the next phase
!python scripts/colab_run_phases.py --phase 1-4 --no-clone

## Step 4 — Benchmark

Check model capability after training. Use `--dual` to see guarded vs model-only scores.

After Phase 1–4 you should see model-only scores jump from ~43% to ~65%+.

In [ ]:
import glob, os

# Find latest checkpoint
ckpt_dirs = sorted(
    glob.glob('/content/checkpoints/yaya-125m-curriculum/**/*.pt', recursive=True),
    key=os.path.getmtime, reverse=True
)
if ckpt_dirs:
    latest_dir = os.path.dirname(ckpt_dirs[0])
    print(f'Benchmarking: {latest_dir}')
    !python scripts/benchmark.py --checkpoint {latest_dir} --dual
else:
    print('No checkpoint found — run training first.')

## Step 5 — Test Yaya directly

Quick interactive test of the trained model.

In [ ]:
import torch, glob, os, sys
sys.path.insert(0, '/content/miss-yaya/yaya-ai')

from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer, ASSISTANT_TOKEN
from src.inference.generator import TextGenerator, GenerationConfig

# Find latest checkpoint
ckpt_dirs = sorted(
    glob.glob('/content/checkpoints/yaya-125m-curriculum/**/*.pt', recursive=True),
    key=os.path.getmtime, reverse=True
)
if not ckpt_dirs:
    print('No checkpoint found — run training first.')
else:
    ckpt_path = ckpt_dirs[0]
    print(f'Loading: {ckpt_path}')
    device = 'cuda'
    model = YayaForCausalLM(load_model_config('configs/model/yaya_125m.yaml')).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state_dict', state))
    model.eval()

    tok = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    gen = TextGenerator(model, tok, device=device)
    cfg = GenerationConfig(max_new_tokens=200, temperature=0.7, top_p=0.9, repetition_penalty=1.5)

    SYSTEM = 'You are Yaya, a helpful, honest, and friendly AI assistant.'

    TESTS = [
        'What is the capital of Kenya?',
        'What is 15% of 240?',
        'Who are you?',
        'What is the Swahili word for water?',
        'If all cats have four legs and Whiskers is a cat, does Whiskers have four legs?',
    ]

    for q in TESTS:
        msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': q}]
        prompt = tok.format_chat(msgs) + '\n' + ASSISTANT_TOKEN + '\n'
        ans = gen.generate(prompt, config=cfg).strip()
        print(f'Q: {q}')
        print(f'A: {ans[:150]}')
        print()

## Step 6 — Download checkpoint (optional)

Download the latest checkpoint to your local machine, or it stays on Drive + HF Hub.

In [ ]:
import glob, shutil, os
from google.colab import files

ckpts = sorted(
    glob.glob('/content/checkpoints/yaya-125m-curriculum/**/*.pt', recursive=True),
    key=os.path.getmtime, reverse=True
)
if ckpts:
    latest_dir = os.path.dirname(ckpts[0])
    zip_path = '/content/yaya-125m-curriculum.zip'
    shutil.make_archive('/content/yaya-125m-curriculum', 'zip', latest_dir)
    print(f'Zipping {latest_dir}...')
    files.download(zip_path)
else:
    print('No checkpoint to download.')